In [ ]:
# =========================
# STEP 1: Import libraries and define file paths
# Purpose: Set up environment and input/output locations
# =========================

import os
import pandas as pd
import numpy as np
from datetime import datetime

# Input files
epc_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
schedules_path = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/schedules.csv"

# Base folder containing building models
base_models_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

# Load EPC dataset
epc_df = pd.read_csv(epc_path)

# Load schedules
schedules_df = pd.read_csv(schedules_path)

print("Step 1 complete: data loaded")
print(epc_df.head())
print(schedules_df.head())

Step 1 complete: data loaded
   BUILDING_REFERENCE_NUMBER  OSG_REFERENCE_NUMBER              ADDRESS1  \
0                 1001829067           122009933.0  19 CULCREUCH AVENUE    
1                 1001951858           122010025.0             GLENVIEW    
2                 1001829071           122009868.0  22 CULCREUCH AVENUE    
3                 1234761001           122009968.0    1 MENZIES TERRACE    
4                 1001991633           122009892.0       50 MAIN STREET    

  ADDRESS2  ADDRESS3 POSTCODE   LATITUDE  LONGITUDE INSPECTION_DATE  \
0  FINTRY   GLASGOW   G63 0YB  56.055692  -4.223337         9/29/25   
1  FINTRY   GLASGOW   G63 0XJ  56.052824  -4.220640         9/19/25   
2  FINTRY   GLASGOW   G63 0YB  56.055503  -4.223691         7/29/25   
3  FINTRY   GLASGOW   G63 0YJ  56.058427  -4.224838         7/22/25   
4  FINTRY   GLASGOW   G63 0XF  56.054738  -4.228307         7/17/25   

         TYPE_OF_ASSESSMENT  ...             ROOF_CONS  WINDOW_CONS  \
0  RdSAP, existi

In [2]:
# =========================
# STEP 2 (REPLACEMENT): Clean schedule extraction + FIXED time vector
# Purpose: Force correct half-hour ordering starting at 00:30:00
# =========================

import numpy as np
import pandas as pd

schedules_df = pd.read_csv(schedules_path)

# ---- FIXED TIME VECTOR (DO NOT RELY ON CSV ORDERING) ----
time_vector = []

start_minutes = 30  # 00:30 start
for i in range(48):
    total_minutes = (start_minutes + i * 30) % (24 * 60)
    hh = total_minutes // 60
    mm = total_minutes % 60
    time_vector.append(f"{hh:02d}:{mm:02d}:00")

# Helper to extract column safely
def get_schedule_col(col_name):
    return schedules_df[col_name].values

# Build lookup dictionary
schedule_map = {
    1: {
        "wd": get_schedule_col("Appliance_1_occupant_with_light_wd"),
        "we": get_schedule_col("Appliance_1_occupant_with_light_we"),
    },
    2: {
        "wd": get_schedule_col("Appliance_2_occupant_with_light_wd"),
        "we": get_schedule_col("Appliance_2_occupant_with_light_we"),
    },
    3: {
        "wd": get_schedule_col("Appliance_3_occupant_with_light_wd"),
        "we": get_schedule_col("Appliance_3_occupant_with_light_we"),
    },
    4: {
        "wd": get_schedule_col("Appliance_4_occupant_with_light_wd"),
        "we": get_schedule_col("Appliance_4_occupant_with_light_we"),
    },
}

print("Step 2 complete: FIXED time vector created")
print(time_vector[:5], "...", time_vector[-5:])

Step 2 complete: FIXED time vector created
['00:30:00', '01:00:00', '01:30:00', '02:00:00', '02:30:00'] ... ['22:00:00', '22:30:00', '23:00:00', '23:30:00', '00:00:00']


In [3]:
# =========================
# STEP 3: Build 2026 calendar pattern
# Purpose: Create weekday/weekend flag for all 365 days (non-leap year)
# =========================

year = 2026

dates = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31", freq="D")

# True = weekend, False = weekday
is_weekend = np.array([d.weekday() >= 5 for d in dates])  # Saturday=5, Sunday=6

print("Step 3 complete: calendar generated")
print("Total days:", len(dates))
print("Weekend days:", is_weekend.sum())

Step 3 complete: calendar generated
Total days: 365
Weekend days: 104


In [4]:
# =========================
# STEP 4 (REPLACEMENT): Annual series generation (uses fixed time vector)
# Purpose: Ensure correct 48-step daily cycle alignment
# =========================

def generate_building_series(occupants):
    """
    Returns:
        time_col (list): repeated 48-step half-hour vector (starts 00:30)
        power_col (list): 17520 values for full year
    """

    occupants = int(np.clip(occupants, 1, 4))

    full_power = []

    for day_idx in range(len(dates)):
        day_type = "we" if is_weekend[day_idx] else "wd"
        daily_profile = schedule_map[occupants][day_type]

        # ensure exactly 48 values per day
        if len(daily_profile) != 48:
            raise ValueError("Schedule column does not contain 48 half-hour values")

        full_power.extend(daily_profile)

    time_col = time_vector * len(dates)

    return time_col, full_power

print("Step 4 complete: generation function fixed")

Step 4 complete: generation function fixed


In [5]:
# =========================
# STEP 5: Loop through all buildings and write outputs
# Purpose: Generate appliance_power_w_lighting.csv for each building folder
# =========================

for _, row in epc_df.iterrows():

    building_id = str(row["BUILDING_REFERENCE_NUMBER"])
    occupants = row["OCCUPANTS_ROUNDED_UP"]

    building_folder = os.path.join(base_models_path, building_id)
    appliances_folder = os.path.join(building_folder, "APPLIANCES")

    if not os.path.exists(appliances_folder):
        print(f"Skipping {building_id}: APPLIANCES folder not found")
        continue

    time_col, power_col = generate_building_series(occupants)

    output_df = pd.DataFrame({
        "Time": time_col,
        "appliance_power": power_col
    })

    output_path = os.path.join(appliances_folder, "appliance_power_w_lighting.csv")
    output_df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

print("Step 5 complete: all buildings processed")

Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/APPLIANCES/appliance_power_w_lighting.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951858/APPLIANCES/appliance_power_w_lighting.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829071/APPLIANCES/appliance_power_w_lighting.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234761001/APPLIANCES/appliance_power_w_lighting.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001991633/APPLIANCES/appliance_power_w_lighting.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664929/APPLIANCES/appliance_power_w_lighting.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829059/APPLIANCES/appliance_power_w_lighting.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063639/APPLIANCES/appliance_power_w_lighting.csv
Saved: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234761000/APPLIANCES/appliance_power_w_light